# SAGE-SR: Sentinel-2 Super-Resolution with DiffFuSR

This notebook performs 4× Super-Resolution on Sentinel-2 L2A satellite multispectral imagery (10 m → 2.5 m) using **DiffFuSR** (Diffusion Fusion Super-Resolution) with pretrained weights on WorldStrat.

### Highlights:
- Compatible with both **Kaggle Notebooks** (GPU enabled) and **Google Colab**.
- Fetches real Sentinel-2 Level-2A imagery over 5 diverse Indian geographic zones (Ladakh mountain range, Ludhiana agricultural belt, central Bangalore urban core, New Delhi India Gate, Yamuna Expressway).
- Performs 4× upscale using degradation-aware diffusion models.
- Produces comparative visual analysis (LR vs. Bicubic Baseline vs. DiffFuSR 4x) along with PSNR and SSIM validation metrics.

**Team:** ThunderBoltz (Smart India Hackathon 2026 - SIH26142)


In [ ]:
# =============================================================================
# DiffFuSR (Diffusion Fusion Super-Resolution) on 5 Indian areas — Colab Cell
# Sentinel-2 L2A RGB 256x256 -> 1024x1024 (4x)
# =============================================================================

import os, sys, glob, types, subprocess, importlib.util

# -----------------------------------------------------------------------------
# 0. Clean kernel state: undo patched torch.load and inject Python 3.12 `imp`
# -----------------------------------------------------------------------------
import torch

while hasattr(torch.load, "__wrapped__"):
    torch.load = torch.load.__wrapped__

_orig_torch_load = torch.load

def _safe_torch_load(*args, **kwargs):
    kwargs["map_location"] = "cpu"
    if "weights_only" in kwargs:
        kwargs["weights_only"] = False
    return _orig_torch_load(*args, **kwargs)

_safe_torch_load.__wrapped__ = _orig_torch_load
torch.load = _safe_torch_load

def _load_source(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

if "imp" not in sys.modules:
    imp_shim = types.ModuleType("imp")
    imp_shim.load_source = _load_source
    sys.modules["imp"] = imp_shim
    print("✅ Patched `imp` module.")

# -----------------------------------------------------------------------------
# 1. System Setup: Environment Detection & Repository Setup
# -----------------------------------------------------------------------------
import shutil

# Robust environment detection (Kaggle vs Colab vs Local)
if os.path.exists("/kaggle/working"):
    BASE_DIR = "/kaggle/working"
elif os.path.exists("/content"):
    BASE_DIR = "/content"
else:
    BASE_DIR = os.path.abspath(".")

print(f"Workspace directory: {BASE_DIR}")
REPO_DIR = os.path.join(BASE_DIR, "DiffFuSR")

# Verify git-lfs
if shutil.which("git-lfs") is None:
    print("Installing git-lfs...")
    subprocess.run(["apt-get", "install", "-y", "--no-install-recommends", "git-lfs"], check=False)
subprocess.run(["git", "lfs", "install"], check=False)

# Check if directory already exists or is incomplete
if os.path.exists(REPO_DIR) and not os.path.exists(os.path.join(REPO_DIR, ".git")):
    print(f"Cleaning up incomplete directory at {REPO_DIR}...")
    shutil.rmtree(REPO_DIR, ignore_errors=True)

if not os.path.exists(REPO_DIR):
    print("Cloning DiffFuSR repository...")
    clone_res = subprocess.run(
        ["git", "clone", "https://github.com/NorskRegnesentral/DiffFuSR.git", REPO_DIR],
        capture_output=True,
        text=True
    )
    if clone_res.returncode != 0:
        print(f"Git clone stderr: {clone_res.stderr}")
        if "Internet" in clone_res.stderr or "Could not resolve host" in clone_res.stderr:
            raise RuntimeError("Internet access appears disabled in Kaggle settings! Please turn on Settings -> Internet (ON).")
        raise RuntimeError(f"Git clone failed: {clone_res.stderr}")
    print("DiffFuSR repository cloned successfully.")

os.chdir(REPO_DIR)
for p in [REPO_DIR, os.path.join(REPO_DIR, "litsr")]:
    if p not in sys.path:
        sys.path.insert(0, p)

CKPT_ROOT = os.path.join(REPO_DIR, "logs")
if not os.path.exists(CKPT_ROOT):
    print("Downloading pretrained checkpoints from Hugging Face...")
    hf_dir = os.path.join(REPO_DIR, "_hf")
    if os.path.exists(hf_dir):
        shutil.rmtree(hf_dir, ignore_errors=True)
    subprocess.run(
        ["git", "clone", "https://huggingface.co/NorskRegnesentralSTI/DiffFuSR", hf_dir],
        check=True,
    )
    subprocess.run(["mv", os.path.join(hf_dir, "logs"), "logs"], check=True)
    shutil.rmtree(hf_dir, ignore_errors=True)
# 2. Dependencies
# -----------------------------------------------------------------------------
PACKAGES = [
    "pystac-client", "rasterio", "scikit-image", "pytorch-lightning",
    "basicsr", "opencv-python-headless", "tifffile", "scipy",
    "pytorch-fid", "thop", "einops", "timm", "kornia", "lpips",
    "gdown", "h5py", "tqdm", "pyyaml", "natsort", "yapf",
]
print("Verifying dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PACKAGES, check=False)

# Install litsr in editable mode
if os.path.exists(os.path.join(REPO_DIR, "litsr")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", os.path.join(REPO_DIR, "litsr")], check=False)

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import Window
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from pystac_client import Client
from pytorch_lightning import seed_everything

# -----------------------------------------------------------------------------
# 3. Comprehensive Auto-Registration: ArchRegistry & ModelRegistry
# -----------------------------------------------------------------------------
import litsr.models
import litsr.archs
from litsr.utils import read_yaml
from litsr.utils.registry import ModelRegistry, ArchRegistry

def register_all_modules_in(search_dir):
    """Recursively imports all .py files in a directory to trigger all decorators."""
    if not os.path.exists(search_dir):
        return
    for root, _, files in os.walk(search_dir):
        for f in files:
            if f.endswith(".py") and not f.startswith("__"):
                fpath = os.path.join(root, f)
                rel_path = os.path.relpath(fpath, REPO_DIR)
                mod_name = os.path.splitext(rel_path)[0].replace(os.sep, ".")
                try:
                    importlib.import_module(mod_name)
                except Exception:
                    try:
                        spec = importlib.util.spec_from_file_location(os.path.splitext(f)[0], fpath)
                        mod = importlib.util.module_from_spec(spec)
                        sys.modules[os.path.splitext(f)[0]] = mod
                        spec.loader.exec_module(mod)
                    except Exception:
                        pass

for d in ["archs", "models", "litsr/archs", "litsr/models"]:
    register_all_modules_in(os.path.join(REPO_DIR, d))

try:
    import train_sr
except Exception:
    pass

print(f"Registered Models: {len(ModelRegistry._obj_map)} classes")
print(f"Registered Archs:  {len(ArchRegistry._obj_map)} classes (includes DegradationEncoder: {'DegradationEncoder' in ArchRegistry._obj_map})")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

CKPT_PATH = os.path.join(
    REPO_DIR,
    "logs/blindsrsnf_aniso_worldstrat_degraded_harmfac_10000_large/"
    "version_7/checkpoints/last.ckpt",
)
if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f"Checkpoint not found at {CKPT_PATH}.")

exp_path = os.path.dirname(os.path.dirname(CKPT_PATH))
config = read_yaml(os.path.join(exp_path, "hparams.yaml"))

model = litsr.models.load_model(config, CKPT_PATH, strict=False)
model = model.to(DEVICE).eval()
print("✅ DiffFuSR WorldStrat SR model loaded successfully.")

# -----------------------------------------------------------------------------
# 4. Areas & helper routines
# -----------------------------------------------------------------------------
CROP = 256
SCALE = 4
TARGET = CROP * SCALE
HALF = 0.02

AREAS = [
    dict(short="Mountain", lon=77.0058, lat=35.0239,
         time="2023-06-01/2023-10-31",
         label="1. Mountain — Ladakh Range, Ladakh"),
    dict(short="Crop area", lon=75.806, lat=30.906,
         time="2023-02-01/2023-04-30",
         label="2. Crop area — Ludhiana wheat belt, Punjab"),
    dict(short="Urban", lon=77.5946, lat=12.9716,
         time="2023-01-01/2024-04-30",
         label="3. Urban — central Bangalore, Karnataka"),
    dict(short="Single building", lon=77.2294, lat=28.6129,
         time="2023-01-01/2024-04-30",
         label="4. Single building — India Gate, New Delhi"),
    dict(short="Highway", lon=77.550, lat=27.600,
         time="2023-01-01/2024-04-30",
         label="5. Highway — Yamuna Expressway, Uttar Pradesh"),
]

catalog = Client.open("https://earth-search.aws.element84.com/v1")

def percentile_stretch(rgb, p_low=2, p_high=98):
    out = np.zeros_like(rgb, dtype=np.uint8)
    for c in range(3):
        lo, hi = np.percentile(rgb[..., c], [p_low, p_high])
        if hi <= lo:
            hi = lo + 1e-6
        out[..., c] = (np.clip((rgb[..., c] - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)
    return out

def read_window(url, size=CROP):
    with rasterio.open(url) as src:
        h, w = src.height, src.width
        y0, x0 = h // 2 - size // 2, w // 2 - size // 2
        return src.read(1, window=Window(x0, y0, size, size))

def fetch_lr_rgb(lon, lat, time_window):
    bbox = [lon - HALF, lat - HALF, lon + HALF, lat + HALF]
    search = catalog.search(
        collections=["sentinel-2-l2a"],
        bbox=bbox,
        datetime=time_window,
        query={"eo:cloud_cover": {"lt": 15}},
        max_items=10,
    )
    items = sorted(search.items(), key=lambda it: it.properties.get("eo:cloud_cover", 100))
    assert items, f"No scene found for ({lat:.3f}, {lon:.3f})"
    item = items[0]
    r = read_window(item.assets["red"].href)
    g = read_window(item.assets["green"].href)
    b = read_window(item.assets["blue"].href)
    rgb = percentile_stretch(np.stack([r, g, b], -1).astype(np.float32))
    return rgb, item

def center_crop(img, size):
    h, w = img.shape[:2]
    y0 = h // 2 - size // 2
    x0 = w // 2 - size // 2
    return img[y0:y0 + size, x0:x0 + size]

# -----------------------------------------------------------------------------
# 5. Inference
# -----------------------------------------------------------------------------
LR_PATCH = 64
SR_PATCH = LR_PATCH * SCALE
seed_everything(42)

results = []
for area in AREAS:
    print(f"\n→ {area['label']}")
    lr_256, item = fetch_lr_rgb(area["lon"], area["lat"], area["time"])
    print(f"   Scene: {item.id} ({item.datetime.date()}, {item.properties.get('eo:cloud_cover'):.2f}% cloud)")

    lr_t = torch.from_numpy(lr_256.astype(np.float32) / 255.0)
    lr_t = lr_t.permute(2, 0, 1).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        rslt = model.test_step_lr_only((lr_t, [area["short"]]))

    sr_np = rslt["log_img_sr"]
    if isinstance(sr_np, torch.Tensor):
        sr_np = sr_np.cpu().numpy().transpose(1, 2, 0)
    sr_1024 = (sr_np * 255).clip(0, 255).astype(np.uint8) if sr_np.max() <= 1.0 else sr_np.astype(np.uint8)

    if sr_1024.shape[0] != TARGET:
        sr_1024 = np.array(Image.fromarray(sr_1024).resize((TARGET, TARGET), Image.BICUBIC))

    lr_patch = center_crop(lr_256, LR_PATCH)
    sr_patch = center_crop(sr_1024, SR_PATCH)

    # --- Bicubic reference at full target resolution -------------------------
    bicubic_256 = np.array(Image.fromarray(lr_256).resize((TARGET, TARGET), Image.BICUBIC))
    # NEW: keep the center crop of the bicubic as well, matching the SR patch
    bicubic_patch = center_crop(bicubic_256, SR_PATCH)

    p = psnr(bicubic_256.astype(np.float32) / 255, sr_1024.astype(np.float32) / 255, data_range=1.0)
    s = ssim(bicubic_256.astype(np.float32) / 255, sr_1024.astype(np.float32) / 255, channel_axis=2, data_range=1.0)
    print(f"   Full-image PSNR vs bicubic: {p:.2f} dB | SSIM: {s:.3f}")

    results.append(dict(
        label=area["label"], short=area["short"],
        lr_patch=lr_patch, bicubic_patch=bicubic_patch, sr_patch=sr_patch,
        psnr=p, ssim=s,
    ))

# -----------------------------------------------------------------------------
# 6. Visualization & export  (LR | Bicubic | DiffFuSR)
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(len(results), 3, figsize=(18, 4.4 * len(results)))

for i, r in enumerate(results):
    # --- Column 0: LR input --------------------------------------------------
    axes[i, 0].imshow(r["lr_patch"], interpolation="nearest")
    axes[i, 0].set_title(f"LR input — {LR_PATCH}×{LR_PATCH} px (10 m)",
                         fontsize=11, fontweight="bold")
    axes[i, 0].set_ylabel(r["label"], fontsize=10, fontweight="bold")
    axes[i, 0].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    # --- Column 1: Bicubic upsample (baseline) -------------------------------
    axes[i, 1].imshow(r["bicubic_patch"])
    axes[i, 1].set_title(f"Bicubic x4 baseline — {SR_PATCH}×{SR_PATCH} px (2.5 m)",
                         fontsize=11, fontweight="bold")
    axes[i, 1].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    # --- Column 2: DiffFuSR SR output ---------------------------------------
    axes[i, 2].imshow(r["sr_patch"])
    axes[i, 2].set_title(
        f"DiffFuSR x4 — {SR_PATCH}×{SR_PATCH} px (2.5 m)   "
        f"[PSNR {r['psnr']:.2f} dB / SSIM {r['ssim']:.3f}]",
        fontsize=11, fontweight="bold"
    )
    axes[i, 2].tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

fig.suptitle(
    "Sentinel-2 L2A → DiffFuSR (WorldStrat SR checkpoint) across five Indian scenes\n"
    "4× upscaling (10 m → 2.5 m) — LR vs Bicubic baseline vs Diffusion SR",
    fontsize=13, fontweight="bold", y=1.00,
)
plt.tight_layout()
output_fig_path = os.path.join(BASE_DIR, "difffusr_5areas_india.png")
plt.savefig(output_fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"\nSaved output figure → {output_fig_path}")
